# ANALÍTICA EMPRESARIAL INTEGRADA
## LABORATORIO DIRIGIDO N.° 10 — Pronóstico de demanda

**Semana 10 · TECSUP 2026-II**
**Docente:** PILAR ROCIO SAYAN MEJIA
**Dataset real:** Online Retail II — UCI Machine Learning Repository (ID 502)
**Objetivo:** construir un pronóstico semanal de ventas, seleccionar modelo y política de cantidad con validación temporal sin información futura, evaluarlos una sola vez y decidir si conviene adoptarlos.

> El notebook termina en una **decisión fundamentada** (adoptar, mantener en evaluación o no adoptar), no solo en el modelo con menor error.

## Trabajo individual y entrega

Esta práctica dirigida es **individual**. Ejecute el notebook en **su propio Google Colab**, verifique los resultados y responda con sus propias interpretaciones.

**Al final de la sesión:**
1. Suba este notebook, ejecutado y con los retos resueltos, a la carpeta `semana10/` de su repositorio personal de GitHub.
2. Conserve la versión en su propio Google Colab y compártala con permiso de lectura.
3. En la **actividad de la semana** presente **ambos URLs**: GitHub y Colab.

**Identificación**
- Nombre: *(PEdro Sebastian Alfieri Arteaga Guerra)*
- Sección: *(C 28)*
- URL de GitHub: *(https://github.com/megumin7w7/analitica-empresarial-integrada)*
- URL de Colab: *(https://colab.research.google.com/drive/1O7IXBjiRAMzHGkNhmFDppsMFEmy1oNvf?usp=sharing)*
- Uso de IA declarado: Se utilizó asistencia de IA para redactar interpretaciones; código y resultados de la ejecución del laboratorio.


# 1. Pregunta de negocio

**¿Cuántas unidades debería planificar la empresa para las próximas semanas, qué riesgo asume si el pronóstico se equivoca y conviene adoptar el modelo?**

Se distinguen cuatro pasos que no deben confundirse:
1. **Pronóstico de ventas:** estimación de las ventas semanales.
2. **Política de cantidad:** regla que convierte el pronóstico en una cantidad objetivo semanal (pronóstico + colchón).
3. **Evaluación económica:** costo de los excedentes y faltantes que produce esa política.
4. **Decisión de adopción:** usar, mantener en evaluación o no usar el modelo, según su resultado frente al baseline.

### Diseño del laboratorio
| Elemento | Definición |
|---|---|
| Unidad temporal | Semana de lunes a domingo, identificada por la fecha del lunes |
| Semanas válidas | Solo semanas completas según la cobertura del archivo |
| Variable objetivo | Unidades vendidas (registradas) por semana de un SKU del Reino Unido |
| Test final | Últimas 8 semanas completas; se evalúa una sola vez |
| Selección del SKU | Criterios calculados solo antes del test |
| Validación | Backtesting rolling-origin con horizonte de 8 semanas dentro del entrenamiento |
| Modelos | Naïve, Naïve estacional (m = 4) y Exponential Smoothing |
| Regla de selección (única) | Candidatos con costo promedio ≤ mínimo × 1,01 (incluye exactamente el 1 %); si el mínimo es 0, solo empatan los costos 0. Desempate: menor MAE, menor desviación del costo, menor complejidad |
| Política de colchón | Multiplicador 0; 0,5; 1 o 1,5 × MAE previo; el MAE previo de cada origen usa solo errores anteriores a ese origen |

## Actividad 1 — Revisión de conceptos

Complete la tabla con sus propias palabras e incluya un ejemplo breve. Si su definición permite responder la pregunta de apoyo, es suficiente; si no, corríjala antes de continuar.

| Concepto | Definición con sus propias palabras y ejemplo | Pregunta de apoyo |
|---|---|---|
| Serie temporal | Observaciones de una misma variable ordenadas en el tiempo a intervalos regulares. Ejemplo: las unidades vendidas de un SKU en cada semana. | ¿Por qué no se pueden barajar sus filas? |
| Semana completa | Semana de lunes a domingo cuyos siete días están cubiertos por el archivo. Ejemplo: la semana que empieza el lunes 2009-12-07.  | ¿Por qué se excluye la primera semana del archivo? |
| Tendencia y estacionalidad | Tendencia: movimiento sostenido del nivel de la serie en el largo plazo. Estacionalidad: patrón que se repite con periodo fijo (por ejemplo, un pico de ventas cada fin de año).  | ¿Basta con que la serie suba y baje para afirmar estacionalidad? |
| Baseline (Naïve y Naïve estacional) | Pronósticos simples de referencia: Naïve repite la última semana observada; Naïve estacional repite la de hace m semanas (aquí m = 4).  | ¿Qué significa que un modelo complejo no supere al Naïve? |
| Exponential Smoothing | Pronóstico que promedia el pasado dando más peso a las semanas recientes (pesos que decaen exponencialmente), con componentes de nivel, tendencia y estacionalidad.  | ¿Más componentes garantizan un mejor pronóstico? |
| Partición temporal (train, validación, test) | División cronológica de la serie: entrenamiento para ajustar, validación para comparar y elegir, test para la evaluación final. Ejemplo: las últimas 8 semanas completas quedan como test.  | ¿Por qué el test se mira una sola vez? |
| Backtesting rolling-origin | Repetir la evaluación moviendo el origen en el tiempo: en cada fold se entrena con todo lo anterior y se pronostica el bloque siguiente.  | ¿Qué aporta frente a un único holdout? |
| MAE, RMSE y WAPE | MAE: promedio del error absoluto, en unidades. RMSE: raíz del error cuadrático medio, castiga más los errores grandes. WAPE: suma de errores absolutos dividida entre la suma de ventas reales, expresada como porcentaje.  | ¿Cuál traduce el error directamente a unidades? |
| Costo asimétrico del error | El exceso y el faltante no cuestan lo mismo. Con los supuestos del laboratorio: £0,40 por unidad excedente y £1,50 por unidad faltante.  | ¿Por qué dos errores de 20 unidades pueden costar distinto? |
| Pronóstico, política de cantidad y evaluación económica | Pronóstico: estimación de las ventas. Política de cantidad: regla que lo convierte en una cantidad objetivo (pronóstico más colchón). Evaluación económica: costo de los excedentes y faltantes que produce esa política.  | ¿La cantidad objetivo es lo que hay que comprar? |

In [1]:
%pip install -q ucimlrepo statsmodels plotly scikit-learn openpyxl

In [2]:
import os
import sys
import warnings
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import sklearn
import statsmodels

from ucimlrepo import fetch_ucirepo
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from sklearn.metrics import mean_absolute_error, mean_squared_error

# No se ocultan advertencias globalmente: las de cada ajuste se registran y se revisan al final.
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)
VERSIONES = {"python": sys.version.split()[0], "pandas": pd.__version__, "numpy": np.__version__,
             "statsmodels": statsmodels.__version__, "scikit-learn": sklearn.__version__}
print("Versiones:", VERSIONES)

Versiones: {'python': '3.13.16', 'pandas': '2.2.3', 'numpy': '2.1.3', 'statsmodels': '0.15.0', 'scikit-learn': '1.6.1'}


# 2. Descarga de datos reales y trazabilidad

Se intenta primero la API oficial de UCI; si falla, el archivo ZIP oficial, del que se leen y concatenan sus **dos hojas**; si ambas fallan, se sube manualmente el archivo descargado desde https://archive.ics.uci.edu/dataset/502. La celda registra fuente, hojas, dimensiones, rango de fechas y versiones.

> Que otra copia tenga el mismo número de filas **no demuestra** que sea idéntica. Con otras versiones de las bibliotecas, los ajustes de Exponential Smoothing pueden producir cifras ligeramente distintas.

In [3]:
from io import BytesIO
from urllib.request import urlopen
from zipfile import ZipFile

def leer_dos_hojas(fuente_excel):
    hojas = pd.read_excel(fuente_excel, sheet_name=None)  # "Year 2009-2010" y "Year 2010-2011"
    return pd.concat(hojas.values(), ignore_index=True), list(hojas)

df, fuente, hojas_leidas = None, None, None
try:
    retail = fetch_ucirepo(id=502)
    df, fuente, hojas_leidas = retail.data.original.copy(), "API ucimlrepo (UCI ID 502)", "tabla única entregada por la API"
except Exception as exc:
    print("API ucimlrepo no disponible:", type(exc).__name__)

if df is None:
    try:
        url = "https://archive.ics.uci.edu/static/public/502/online+retail+ii.zip"
        with urlopen(url, timeout=120) as respuesta:
            blob = respuesta.read()
        with ZipFile(BytesIO(blob)) as z:
            nombre = next(n for n in z.namelist() if n.lower().endswith(".xlsx"))
            df, hojas_leidas = leer_dos_hojas(BytesIO(z.read(nombre)))
        fuente = "ZIP oficial de UCI"
    except Exception as exc:
        print("ZIP oficial no disponible:", type(exc).__name__)

if df is None:
    locales = sorted(f for f in os.listdir(".")
                     if "online" in f.lower() and "retail" in f.lower() and f.lower().endswith((".xlsx", ".csv")))
    if not locales:
        try:
            from google.colab import files
            print("Suba el archivo online_retail_II.xlsx descargado desde UCI.")
            locales = list(files.upload().keys())
        except ImportError:
            raise FileNotFoundError("Coloque online_retail_II.xlsx en la carpeta de trabajo y vuelva a ejecutar.")
    archivo = locales[0]
    if archivo.lower().endswith(".xlsx"):
        df, hojas_leidas = leer_dos_hojas(archivo)
    else:
        df = pd.read_csv(archivo, dtype={"Invoice": str, "InvoiceNo": str, "StockCode": str})
        hojas_leidas = "no aplica (CSV)"
    fuente = f"archivo local: {archivo}"

fechas_raw = pd.to_datetime(df[[c for c in df.columns if "date" in str(c).lower()][0]], errors="coerce")
print("Fuente utilizada:", fuente)
print("Hojas:", hojas_leidas)
print("Dimensión:", df.shape)
print("Rango de fechas:", fechas_raw.min(), "→", fechas_raw.max())
print("Ejecución:", pd.Timestamp.now(tz="UTC").isoformat())
print("Versiones:", VERSIONES)
display(df.head())

API ucimlrepo no disponible: DatasetNotFoundError
Fuente utilizada: ZIP oficial de UCI
Hojas: ['Year 2009-2010', 'Year 2010-2011']
Dimensión: (1067371, 8)
Rango de fechas: 2009-12-01 07:45:00 → 2011-12-09 12:50:00
Ejecución: 2026-10-26T04:24:11.894574+00:00
Versiones: {'python': '3.13.16', 'pandas': '2.2.3', 'numpy': '2.1.3', 'statsmodels': '0.15.0', 'scikit-learn': '1.6.1'}


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [4]:
def find_col(columns, candidates):
    normalized = {str(c).lower().replace(" ", "").replace("_", ""): c for c in columns}
    for cand in candidates:
        key = cand.lower().replace(" ", "").replace("_", "")
        if key in normalized:
            return normalized[key]
    return None

mapping = {
    "InvoiceNo": find_col(df.columns, ["InvoiceNo", "Invoice"]),
    "StockCode": find_col(df.columns, ["StockCode"]),
    "Description": find_col(df.columns, ["Description"]),
    "Quantity": find_col(df.columns, ["Quantity"]),
    "InvoiceDate": find_col(df.columns, ["InvoiceDate"]),
    "UnitPrice": find_col(df.columns, ["UnitPrice", "Price"]),
    "CustomerID": find_col(df.columns, ["Customer ID", "CustomerID"]),
    "Country": find_col(df.columns, ["Country"]),
}
faltan = [k for k, v in mapping.items() if v is None and k not in ("Description", "CustomerID")]
if faltan:
    raise ValueError(f"No se identificaron columnas obligatorias: {faltan}")
df = df.rename(columns={v: k for k, v in mapping.items() if v is not None})
print(mapping)

{'InvoiceNo': 'Invoice', 'StockCode': 'StockCode', 'Description': 'Description', 'Quantity': 'Quantity', 'InvoiceDate': 'InvoiceDate', 'UnitPrice': 'Price', 'CustomerID': 'Customer ID', 'Country': 'Country'}


# 3. Control de calidad

Los `StockCode` nulos se descartan **antes** de convertir a texto; de lo contrario, la ausencia se convertiría en un producto llamado «nan».

In [5]:
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"], errors="coerce")
df["Quantity"] = pd.to_numeric(df["Quantity"], errors="coerce")
df["UnitPrice"] = pd.to_numeric(df["UnitPrice"], errors="coerce")
print("StockCode nulos descartados:", int(df["StockCode"].isna().sum()))
df = df[df["StockCode"].notna()].copy()
df["StockCode"] = df["StockCode"].astype(str).str.strip()
df = df[df["StockCode"] != ""]
cancel = df["InvoiceNo"].astype(str).str.upper().str.startswith("C")

print("Fecha mínima:", df["InvoiceDate"].min(), "·", df["InvoiceDate"].min().day_name())
print("Fecha máxima:", df["InvoiceDate"].max(), "·", df["InvoiceDate"].max().day_name())
print("Cancelaciones:", int(cancel.sum()))
print("Quantity <= 0:", int((df["Quantity"] <= 0).sum()))
print("UnitPrice <= 0:", int((df["UnitPrice"] <= 0).sum()))
print("Fechas nulas:", int(df["InvoiceDate"].isna().sum()))

clean = df.loc[(~cancel) & df["InvoiceDate"].notna() & (df["Quantity"] > 0) &
               (df["UnitPrice"] > 0) & df["Country"].notna()].copy()
uk = clean[clean["Country"].astype(str).str.upper().eq("UNITED KINGDOM")].copy()
print("Filas válidas UK:", len(uk), "· SKUs UK:", uk["StockCode"].nunique())

StockCode nulos descartados: 0
Fecha mínima: 2009-12-01 07:45:00 · Tuesday
Fecha máxima: 2011-12-09 12:50:00 · Friday
Cancelaciones: 19494
Quantity <= 0: 22950
UnitPrice <= 0: 6207
Fechas nulas: 0
Filas válidas UK: 958501 · SKUs UK: 4905


### Pregunta 1
¿Por qué no sería correcto sumar directamente `Quantity` sin eliminar cancelaciones y cantidades no positivas?

**Respuesta:**

Porque `Quantity` mezcla eventos que no son ventas. Las facturas canceladas (Invoice que empieza con «C») traen cantidades negativas: son devoluciones o anulaciones de ventas que se hicieron antes, en otra semana. Los registros con `Quantity ≤ 0` incluyen además ajustes de inventario, mermas y correcciones, y los de `UnitPrice ≤ 0` son ajustes o regalos sin precio. En este archivo son cifras grandes: 19494 cancelaciones, 22950 filas con `Quantity ≤ 0` y 6207 con `UnitPrice ≤ 0`, sobre 1067371 filas en total.

Si se suma todo directamente, se restan unidades de la semana equivocada: algunas semanas quedarían subestimadas, otras con valores artificialmente bajos o hasta negativos, y el modelo aprendería ruido contable en lugar del patrón de ventas. Como el objetivo del laboratorio es pronosticar **unidades vendidas registradas**, se filtran los registros para que cada unidad corresponda a una venta efectiva (quedan 958501 filas válidas del Reino Unido y 4905 SKUs). La consecuencia es que la serie representa ventas brutas, no netas de devoluciones.

# 4. Semanas completas y partición temporal

Las semanas van de **lunes a domingo** y se identifican por la fecha del lunes. Una semana es **completa** solo si el archivo cubre sus siete días: se excluyen la primera semana si el archivo no empieza en lunes y la última si no termina en domingo. Las **últimas 8 semanas completas** se reservan como test final.

Algunas semanas no tienen **ninguna venta registrada** en los datos filtrados. El cierre de la tienda es una **posible explicación no verificada**: los datos no lo confirman.

In [6]:
def lunes(fechas):
    fechas = pd.to_datetime(fechas)
    return fechas.dt.normalize() - pd.to_timedelta(fechas.dt.weekday, unit="D")

uk["week"] = lunes(uk["InvoiceDate"])
inicio_archivo, fin_archivo = df["InvoiceDate"].min(), df["InvoiceDate"].max()
lunes_inicio = inicio_archivo.normalize() - pd.Timedelta(days=inicio_archivo.weekday())
lunes_fin = fin_archivo.normalize() - pd.Timedelta(days=fin_archivo.weekday())
primera_completa = lunes_inicio if inicio_archivo.weekday() == 0 else lunes_inicio + pd.Timedelta(weeks=1)
ultima_completa = lunes_fin if fin_archivo.weekday() == 6 else lunes_fin - pd.Timedelta(weeks=1)

semanas = pd.date_range(primera_completa, ultima_completa, freq="W-MON")
TEST_H = 8
semanas_test = semanas[-TEST_H:]
semanas_train = semanas[:-TEST_H]

registros = uk.groupby("week").size().reindex(semanas, fill_value=0)
semanas_sin_registro = registros[registros == 0].index

print("Semanas excluidas por incompletas:", lunes_inicio.date(), "y", lunes_fin.date())
print("Semanas completas:", len(semanas), "·", semanas[0].date(), "→", semanas[-1].date())
print("Entrenamiento:", len(semanas_train), "semanas ·", semanas_train[0].date(), "→", semanas_train[-1].date())
print("Test final:", len(semanas_test), "semanas ·", semanas_test[0].date(), "→", semanas_test[-1].date())
print("Semanas sin ventas registradas en los datos filtrados (cierre: explicación posible no verificada):",
      [d.date() for d in semanas_sin_registro])

Semanas excluidas por incompletas: 2009-11-30 y 2011-12-05
Semanas completas: 104 · 2009-12-07 → 2011-11-28
Entrenamiento: 96 semanas · 2009-12-07 → 2011-10-03
Test final: 8 semanas · 2011-10-10 → 2011-11-28
Semanas sin ventas registradas en los datos filtrados (cierre: explicación posible no verificada): [datetime.date(2009, 12, 28), datetime.date(2010, 12, 27)]


### Pregunta 2
¿Por qué una partición aleatoria con `train_test_split` sería incorrecta en este problema?

**Respuesta:**

Porque una partición aleatoria mezcla semanas del futuro dentro del entrenamiento: el modelo vería semanas posteriores a las que luego tiene que pronosticar. Eso es **fuga de información**, y además rompe el orden temporal del que dependen la tendencia, la estacionalidad y la relación entre semanas vecinas (una semana de test queda rodeada de semanas de entrenamiento, y el modelo casi la "interpola"). El error medido saldría optimista. En producción solo se conoce el pasado, por eso la evaluación debe imitarlo: entrenar con lo anterior y probar con lo posterior, como hacen la partición cronológica y el backtesting rolling-origin.

# 5. Selección reproducible del producto (solo con el entrenamiento)

Criterios, calculados **exclusivamente con las semanas de entrenamiento**:
1. **Periodo con ventas registradas cercano al entrenamiento completo:** primera venta dentro de las **4 primeras** semanas y última venta dentro de las **4 últimas**. El criterio tolera hasta 3 semanas iniciales o finales sin ventas, que luego se completan con cero aunque podrían corresponder a un periodo sin disponibilidad. **No verifica inventario.**
2. **Cobertura:** ventas registradas en al menos el 90 % de las semanas de entrenamiento con algún registro en la tienda.
3. Entre los elegibles, el de **mayor cantidad de unidades**; desempate por más semanas con ventas.

Se comprueba que haya al menos un producto elegible (y al menos dos para el Reto 3).

In [7]:
uk_train = uk[uk["week"].isin(semanas_train)]
semanas_con_registro_train = [w for w in semanas_train if w not in set(semanas_sin_registro)]

cobertura = (uk_train.groupby(["StockCode", "week"])["Quantity"].sum().reset_index()
             .groupby("StockCode")
             .agg(semanas_con_venta=("week", "nunique"), unidades=("Quantity", "sum"),
                  primera_venta=("week", "min"), ultima_venta=("week", "max"))
             .reset_index())
cobertura["cobertura"] = cobertura["semanas_con_venta"] / len(semanas_con_registro_train)
elegibles = cobertura[
    (cobertura["primera_venta"] <= semanas_train[3]) &      # dentro de las 4 primeras semanas
    (cobertura["ultima_venta"] >= semanas_train[-4]) &      # dentro de las 4 últimas semanas
    (cobertura["cobertura"] >= 0.90)
].sort_values(["unidades", "semanas_con_venta"], ascending=False).reset_index(drop=True)

if len(elegibles) == 0:
    raise ValueError("Ningún SKU cumple los criterios: revise la carga y la limpieza antes de continuar.")
print("SKUs elegibles:", len(elegibles), "· suficientes para el Reto 3:", len(elegibles) >= 2)
SKU = elegibles.loc[0, "StockCode"]
desc = uk.loc[uk["StockCode"] == SKU, "Description"].dropna().astype(str).mode().iloc[0]
print("SKU seleccionado:", SKU, "·", desc)
display(elegibles.head(5))

SKUs elegibles: 389 · suficientes para el Reto 3: True
SKU seleccionado: 84077 · WORLD WAR 2 GLIDERS ASSTD DESIGNS


,StockCode,semanas_con_venta,unidades,primera_venta,ultima_venta,cobertura
0,84077,93,88273,2009-12-07,2011-10-03,0.989362
1,85123A,94,80903,2009-12-07,2011-10-03,1.000000
2,85099B,94,79134,2009-12-07,2011-10-03,1.000000
3,21212,94,67878,2009-12-07,2011-10-03,1.000000
4,17003,88,67514,2009-12-07,2011-10-03,0.936170


# 6. Serie semanal de ventas registradas

**Qué representa un cero.** Un cero indica que **no hay ventas registradas** del SKU en esa semana dentro de los datos filtrados. Los datos no permiten distinguir entre:
- **disponibilidad de inventario** con demanda nula;
- **demanda no satisfecha** por falta de inventario (no queda registrada);
- **ausencia de registros** (por ejemplo, semanas sin ninguna venta registrada en la tienda).

Completar con cero es un **supuesto sobre las ventas observadas**: no demuestra demanda cero ni disponibilidad continua. Por eso el laboratorio pronostica **ventas registradas**, que solo aproximan la demanda.

In [8]:
serie = (uk[uk["StockCode"] == SKU].groupby("week")["Quantity"].sum()
         .reindex(semanas, fill_value=0).astype(float))
serie.name = "unidades"
train = serie.loc[semanas_train]
test = serie.loc[semanas_test]

ceros = serie[serie == 0].index
print("Semanas de la serie:", len(serie), "· semanas sin ventas registradas del SKU:", len(ceros))
print("  coinciden con semanas sin ningún registro en la tienda:", [d.date() for d in ceros if d in set(semanas_sin_registro)])
print("  semanas con registros en la tienda pero sin ventas del SKU:", [d.date() for d in ceros if d not in set(semanas_sin_registro)])
print("Unidades en entrenamiento:", int(train.sum()))

Semanas de la serie: 104 · semanas sin ventas registradas del SKU: 3
  coinciden con semanas sin ningún registro en la tienda: [datetime.date(2009, 12, 28), datetime.date(2010, 12, 27)]
  semanas con registros en la tienda pero sin ventas del SKU: [datetime.date(2011, 1, 17)]
Unidades en entrenamiento: 88273


# 7. Exploración (solo con el entrenamiento)

La exploración que orienta decisiones usa únicamente el entrenamiento: mirar el test antes de elegir el modelo sería una fuga de información. En este laboratorio las medias móviles de las **últimas 4 y 8 semanas** se usan para **explorar** el nivel; no se evalúan como método de pronóstico (podrían usarse como tal, pero no forman parte de esta comparación).

In [9]:
explora = pd.DataFrame({"ventas": train})
explora["media_movil_4"] = explora["ventas"].rolling(4).mean()
explora["media_movil_8"] = explora["ventas"].rolling(8).mean()
fig = go.Figure()
for col, nombre in [("ventas", "Ventas registradas"), ("media_movil_4", "Media móvil 4"), ("media_movil_8", "Media móvil 8")]:
    fig.add_trace(go.Scatter(x=explora.index, y=explora[col], name=nombre))
fig.update_layout(title=f"Entrenamiento · SKU {SKU} · {desc}", xaxis_title="Semana (lunes)", yaxis_title="Unidades")
fig.show()

perfil = (explora.assign(semana_anio=explora.index.isocalendar().week.astype(int))
          .groupby("semana_anio")["ventas"].mean().reset_index())
fig = px.line(perfil, x="semana_anio", y="ventas", markers=True,
              title="Perfil promedio por semana del año (entrenamiento)",
              labels={"semana_anio": "Semana del año", "ventas": "Ventas promedio"})
fig.show()

anual = train.groupby(train.index.year).agg(["count", "mean", "median", "max"])
display(anual.round(1))

,count,mean,median,max
2009,4,493.0,384.0,1204.0
2010,52,1034.4,608.0,7063.0
2011,40,812.8,558.0,3216.0


### Pregunta 3
Identifique evidencia de tendencia, estacionalidad o quiebres en el entrenamiento. Diferencie lo que observa de lo que todavía necesita comprobar, y explique qué representa un cero en la serie.

**Respuesta:**

**Lo que se observa (SKU 84077, entrenamiento de 96 semanas).**

- *Tendencia:* no hay evidencia de una tendencia sostenida. La media anual baja de 1034,4 unidades en 2010 (52 semanas) a 812,8 en 2011 (40 semanas), pero la mediana casi no cambia (608 → 558) y la media de 2010 está inflada por picos (máximo 7063). Además la comparación no es equivalente: 2011 solo llega hasta el 3-oct y no incluye el tramo alto de fin de año. Las medias móviles de 4 y 8 semanas se mueven sobre todo entre unas 500 y 1000 unidades, con una subida visible en octubre de 2010 a enero de 2011 (media móvil de 8 semanas cerca de 2000) y un retorno al nivel anterior después.
- *Estacionalidad:* el perfil por semana del año muestra picos en las semanas 44 (≈5300 unidades), 48 (≈7000) y 40 (≈2200). Pero esas tres semanas del tramo de fin de año aparecen **una sola vez** en el entrenamiento (2010); 2009 solo aporta 4 semanas de diciembre. Con una sola observación por semana, lo que veo es un **posible** patrón de fin de año que todavía necesita comprobarse con más ciclos; no basta para afirmar estacionalidad. El periodo de 4 semanas del Naïve estacional es un supuesto de modelado, no una propiedad demostrada.
- *Quiebres y picos:* el máximo (7063) es unas 12 veces la mediana del entrenamiento (584,5). Los picos son aislados y la serie vuelve a su nivel habitual en las semanas siguientes, así que parecen picos puntuales y no un cambio de nivel que se mantiene.

**Qué representa un cero.** Hay 3 semanas en cero. Un cero significa que **no hay ventas registradas** del SKU en esa semana en los datos filtrados, no que la demanda fuera cero: no distingue entre inventario disponible sin demanda, demanda no satisfecha por falta de stock (que no queda registrada) y ausencia de registros. Dos de los ceros (semanas del 2009-12-28 y 2010-12-27) coinciden con semanas sin ningún registro en la tienda, lo que sugiere un cierre de fin de año (explicación posible, no verificada); el tercero (2011-01-17) tiene registros en la tienda pero ninguna venta del SKU. Completar con cero es un supuesto sobre las ventas observadas.

# 8. Modelos, cantidad operativa, costo y regla de selección

- **Naïve:** repite la última semana observada.
- **Naïve estacional (m = 4):** repite la semana de hace 4 semanas. El periodo de 4 semanas es un **supuesto de modelado** (ciclo operativo corto), no una propiedad comprobada del negocio; con 96 semanas de entrenamiento no hay suficientes años para estimar con confianza un ciclo anual de 52 semanas.
- **Exponential Smoothing:** nivel y tendencia aditivos con estacionalidad aditiva m = 4. Si un entrenamiento tuviera menos de 2 × 4 + 8 = 16 semanas, se simplifica a nivel y tendencia (documentado; se informa si ocurre).

**Cantidad operativa:** todo pronóstico se convierte en `max(redondeo(pronóstico), 0)` y **todas** las métricas y costos se calculan sobre esa cantidad.

**Costo base (supuesto académico, no proviene de UCI):** £0,40 por unidad excedente y £1,50 por unidad faltante.

**Regla única de selección** (se usa para el modelo, la política de colchón, el Reto 1 y el Reto 3):
1. Candidatos: costo promedio de validación ≤ mínimo × 1,01 (**incluye** exactamente el 1 %; se admite una holgura de 1e-9 por redondeo numérico). Si el mínimo es 0, solo son candidatos los costos iguales a 0.
2. Desempate entre candidatos: menor MAE promedio, luego menor desviación del costo y, al final, menor complejidad.

In [10]:
SEASON = 4
OVER_COST, SHORT_COST = 0.40, 1.50
MODELOS = ["Naive", "SeasonalNaive", "ExpSmoothing"]   # en orden de complejidad
TOLERANCIA = 0.01
registro_ajustes = []

def ajustar_es(y, etiqueta):
    """Ajusta Exponential Smoothing y registra todas las advertencias del ajuste."""
    if len(y) >= 2 * SEASON + 8:
        modelo = ExponentialSmoothing(y, trend="add", seasonal="add", seasonal_periods=SEASON,
                                      initialization_method="estimated")
        especificacion = "nivel + tendencia + estacionalidad aditivas (m = 4)"
    else:
        modelo = ExponentialSmoothing(y, trend="add", seasonal=None, initialization_method="estimated")
        especificacion = "simplificado: nivel + tendencia"
    with warnings.catch_warnings(record=True) as avisos:
        warnings.simplefilter("always")
        ajuste = modelo.fit(optimized=True)
    registro_ajustes.append({"etapa": etiqueta.split(" · ")[0], "ajuste": etiqueta, "n_train": len(y),
                             "especificacion": especificacion, "advertencias": len(avisos),
                             "tipos": "; ".join(sorted({w.category.__name__ for w in avisos})),
                             "mensajes": " | ".join(sorted({str(w.message)[:110] for w in avisos}))})
    return ajuste

def pronosticar(modelo, y, h, etiqueta=""):
    if modelo == "Naive":
        f = np.repeat(y.iloc[-1], h)
    elif modelo == "SeasonalNaive":
        f = np.resize(y.iloc[-SEASON:].values, h)
    elif modelo == "ExpSmoothing":
        f = np.asarray(ajustar_es(y, etiqueta).forecast(h), dtype=float)
    else:
        raise ValueError(modelo)
    return np.maximum(np.round(f), 0)   # cantidad operativa

def evaluar(real, cantidad, over=OVER_COST, short=SHORT_COST):
    real, cantidad = np.asarray(real, float), np.asarray(cantidad, float)
    exceso, faltante = np.maximum(cantidad - real, 0), np.maximum(real - cantidad, 0)
    return {"MAE": mean_absolute_error(real, cantidad),
            "RMSE": mean_squared_error(real, cantidad) ** 0.5,
            "WAPE": np.abs(real - cantidad).sum() / real.sum() if real.sum() > 0 else np.nan,  # no definido si la demanda total es 0
            "unidades_excedentes": exceso.sum(), "unidades_faltantes": faltante.sum(),
            "costo": (exceso * over + faltante * short).sum()}

def resumir(filas, por, orden):
    """Resume los folds por modelo o por política. 'orden' va de la opción más simple a la más compleja."""
    t = (pd.DataFrame(filas).groupby(por)
           .agg(costo_prom=("costo", "mean"), costo_sd=("costo", "std"),
                MAE_prom=("MAE", "mean"), MAE_sd=("MAE", "std"), RMSE_prom=("RMSE", "mean"),
                WAPE_prom=("WAPE", "mean"), excedentes=("unidades_excedentes", "sum"),
                faltantes=("unidades_faltantes", "sum"))
           .reindex(orden))
    t["complejidad"] = range(len(orden))
    return t

def seleccionar(tabla, tol=TOLERANCIA):
    """Regla única de selección (ver el texto de esta sección)."""
    minimo = tabla["costo_prom"].min()
    limite = minimo * (1 + tol)          # si minimo == 0, limite == 0: solo empatan costos 0
    candidatos = tabla[tabla["costo_prom"] <= limite + 1e-9]
    return candidatos.sort_values(["MAE_prom", "costo_sd", "complejidad"], kind="stable").index[0]
print("Funciones listas.")

Funciones listas.


# 9. Backtesting rolling-origin (solo con el entrenamiento)

Cada fold entrena con todo lo anterior a su origen y pronostica las **8 semanas** siguientes, igual que el test final. Se exige al menos 52 semanas de entrenamiento; los bloques de validación no se superponen y el último termina justo antes del test. La celda **comprueba** que ningún fold invada el test.

In [11]:
BT_H, MIN_TRAIN = 8, 52
origenes = list(range(len(train) - BT_H, MIN_TRAIN - 1, -BT_H))[::-1]

def backtest(y, modelos, etiqueta):
    """Pronostica cada bloque de validación desde su origen. Devuelve métricas por fold y cantidades."""
    filas, cantidades = [], {}
    for k, origen in enumerate(origenes, start=1):
        tr, va = y.iloc[:origen], y.iloc[origen:origen + BT_H]
        assert tr.index[-1] < va.index[0] and va.index[-1] < semanas_test[0], "un fold invade el test"
        for m in modelos:
            cantidades[(k, m)] = pronosticar(m, tr, BT_H, f"{etiqueta} · fold {k}")
            filas.append({**evaluar(va, cantidades[(k, m)]), "fold": k, "modelo": m})
    return filas, cantidades

filas_bt, q_bt = backtest(train, MODELOS, "backtesting")
bt = pd.DataFrame(filas_bt)
folds = pd.DataFrame([{"fold": k, "entrenamiento": f"{train.index[0].date()} → {train.index[o - 1].date()}", "n_train": o,
                       "validacion": f"{train.index[o].date()} → {train.index[o + BT_H - 1].date()}"}
                      for k, o in enumerate(origenes, start=1)])
display(folds)
display(bt.round(2))
print("Comprobado: los", len(origenes), "folds terminan antes del test (", semanas_test[0].date(), ").")

,fold,entrenamiento,n_train,validacion
0,1,2009-12-07 → 2010-12-27,56,2011-01-03 → 2011-02-21
1,2,2009-12-07 → 2011-02-21,64,2011-02-28 → 2011-04-18
2,3,2009-12-07 → 2011-04-18,72,2011-04-25 → 2011-06-13
3,4,2009-12-07 → 2011-06-13,80,2011-06-20 → 2011-08-08
4,5,2009-12-07 → 2011-08-08,88,2011-08-15 → 2011-10-03


,MAE,RMSE,WAPE,unidades_excedentes,unidades_faltantes,costo,fold,modelo
0,608.12,1044.62,1.00,0.0,4865.0,7297.5,1,Naive
1,528.88,758.45,0.87,1195.0,3036.0,5032.0,1,SeasonalNaive
2,1599.12,1838.12,2.63,11475.0,1318.0,6567.0,1,ExpSmoothing
3,1111.12,1460.84,0.83,113.0,8776.0,13209.2,2,Naive
4,789.88,1000.51,0.59,1317.0,5002.0,8029.8,2,SeasonalNaive
5,949.12,1147.68,0.71,4251.0,3342.0,6713.4,2,ExpSmoothing
6,723.88,824.92,0.77,4550.0,1241.0,3681.5,3,Naive
7,1248.88,1463.50,1.33,8557.0,1434.0,5573.8,3,SeasonalNaive
8,850.75,960.03,0.91,5384.0,1422.0,4286.6,3,ExpSmoothing
9,391.88,471.10,0.75,47.0,3088.0,4650.8,4,Naive


Comprobado: los 5 folds terminan antes del test ( 2011-10-10 ).


In [12]:
resumen_val = resumir(filas_bt, "modelo", MODELOS)
SELECCIONADO = seleccionar(resumen_val)
display(resumen_val.round(2))
print("MODELO SELECCIONADO EN VALIDACIÓN (fijado antes de mirar el test):", SELECCIONADO)
fig = px.box(bt, x="modelo", y="costo", points="all", title="Costo simulado por fold de validación (£)",
             labels={"modelo": "Modelo", "costo": "Costo por fold (£)"})
fig.show()

,costo_prom,costo_sd,MAE_prom,MAE_sd,RMSE_prom,WAPE_prom,excedentes,faltantes,complejidad
modelo,,,,,,,,,
Naive,6239.22,4296.41,620.2,327.68,819.13,0.75,5469.0,19339.0,0
SeasonalNaive,4950.34,2065.83,666.5,370.02,830.61,0.83,13853.0,12807.0,1
ExpSmoothing,4324.10,2335.11,927.3,420.72,1063.63,1.29,30925.0,6167.0,2


MODELO SELECCIONADO EN VALIDACIÓN (fijado antes de mirar el test): ExpSmoothing


### Pregunta 4
¿Qué modelo se seleccionó en validación y con qué regla? ¿Coinciden el ranking por costo y el ranking por MAE o RMSE? ¿Por qué el ranking por WAPE no aporta un criterio adicional al MAE?

**Respuesta:**

Se seleccionó **Exponential Smoothing** con la **regla única**: son candidatos los modelos con costo promedio de validación dentro del 1 % del mínimo (≤ mínimo × 1,01) y entre ellos se desempata por menor MAE, menor desviación del costo y menor complejidad. Aquí no hizo falta el desempate: Exponential Smoothing tuvo el menor costo (£4312,56) y el siguiente, Naïve estacional (£4950,34), queda muy por encima del límite de £4355,69; fue el único candidato.

*Ranking por costo vs. MAE y RMSE: no coinciden, se invierten.* Por costo el orden es Exponential Smoothing (£4312,56) < Naïve estacional (£4950,34) < Naïve (£6239,22). Por MAE (898,32 / 666,50 / 620,20), por RMSE (1030,27 / 830,61 / 819,13) y por WAPE el orden es el contrario: Naïve < Naïve estacional < Exponential Smoothing. La razón está en los errores por signo: Exponential Smoothing sobreestima (29397 unidades excedentes frente a 6536 faltantes en validación), mientras que Naïve subestima (5469 excedentes frente a 19339 faltantes). El MAE y el RMSE tratan igual sobrar y faltar, pero cada faltante cuesta £1,50 y cada excedente £0,40 (3,75 veces menos), de modo que el modelo con peor error en unidades resulta el más barato.

*WAPE.* Es la suma de errores absolutos dividida entre la suma de las ventas reales. Dentro de un mismo fold todos los modelos se comparan contra los mismos datos reales, así que el denominador es común y WAPE equivale a MAE reescalado: ordena igual que el MAE (0,75 < 0,83 < 1,24) y no agrega un criterio nuevo. Solo cambia la escala; que Exponential Smoothing supere 1 (1,24) indica que su error absoluto fue mayor que las ventas reales de esos bloques, sobre todo en el fold 1 (WAPE 2,63), cuando las ventas eran bajas y el modelo proyectó demasiado alto.

### Pregunta 5
¿Qué tan estable fue cada modelo entre folds? Use la desviación del costo y del MAE.

**Respuesta:**

# 10. Política de colchón calibrada sin información futura (antes del test)

El colchón suma unidades al pronóstico para protegerse de faltantes: **colchón = ⌈multiplicador × MAE previo⌉**. Se comparan los multiplicadores 0; 0,5; 1 y 1,5.

**Regla de calibración (sin errores futuros).** Para cada origen, el *MAE previo* se calcula con el modelo seleccionado en los **2 bloques de 8 semanas inmediatamente anteriores** a ese origen: cada bloque se pronostica entrenando solo con lo anterior a él. Así, el colchón del fold 1 usa solo errores anteriores al fold 1.
- Si no hay historia suficiente (algún bloque quedaría con menos de 16 semanas de entrenamiento), el colchón de ese origen es **0** y se informa. Con 52 semanas mínimas no ocurre en este laboratorio.
- La política (modelo, multiplicador y regla) se elige con la **regla única de selección** y queda **fijada antes del test**.
- Para el test, el MAE previo se calcula con la misma regla desde el final del entrenamiento (solo información previa al test).
- Después de evaluar, la **magnitud** del colchón se actualiza con la historia completa para proyectar, **sin cambiar la regla**.

In [13]:
MULTIPLICADORES = [0, 0.5, 1.0, 1.5]
BLOQUES_CAL = 2

def mae_previo(modelo, y, etiqueta, bloques=BLOQUES_CAL):
    """MAE del modelo en los 'bloques' bloques de 8 semanas que terminan al final de y.
    Solo usa y (información hasta el origen). Devuelve None si la historia no alcanza."""
    errores = []
    for b in range(bloques, 0, -1):
        corte = len(y) - b * BT_H
        if corte < 2 * SEASON + 8:
            return None
        real = y.iloc[corte:corte + BT_H].values
        errores.append(np.abs(real - pronosticar(modelo, y.iloc[:corte], BT_H, f"{etiqueta} · bloque {b}")))
    return float(np.mean(np.concatenate(errores)))

def colchon(mult, mae):
    return 0 if (mae is None or mult == 0) else int(np.ceil(mult * mae))

def comparar_politicas(bloques=BLOQUES_CAL, etiqueta="calibración"):
    filas = []
    for k, origen in enumerate(origenes, start=1):
        tr, va = train.iloc[:origen], train.iloc[origen:origen + BT_H]
        mae_k = mae_previo(SELECCIONADO, tr, f"{etiqueta} · fold {k}", bloques)
        for mult in MULTIPLICADORES:
            c = colchon(mult, mae_k)
            filas.append({**evaluar(va, q_bt[(k, SELECCIONADO)] + c), "fold": k, "multiplicador": mult,
                          "mae_previo": mae_k, "colchon": c,
                          "ultima_semana_calibracion": tr.index[-1].date(), "inicio_validacion": va.index[0].date()})
    return pd.DataFrame(filas)

pol_folds = comparar_politicas()
assert (pol_folds["ultima_semana_calibracion"] < pol_folds["inicio_validacion"]).all(), "colchón con información futura"
resumen_pol = resumir(pol_folds, "multiplicador", MULTIPLICADORES)
MULT = seleccionar(resumen_pol)

display(pol_folds.loc[pol_folds["multiplicador"] == 1.0,
                      ["fold", "ultima_semana_calibracion", "inicio_validacion", "mae_previo"]].round(1))
display(resumen_pol.round(2))
MAE_TEST = mae_previo(SELECCIONADO, train, "calibración test")       # solo entrenamiento
COLCHON_TEST = colchon(MULT, MAE_TEST)
POLITICA = f"{SELECCIONADO} + {MULT:g} × MAE previo"
print("POLÍTICA FIJADA ANTES DEL TEST:", POLITICA)
print(f"Colchón para el test: {COLCHON_TEST} unidades por semana (MAE previo = {MAE_TEST:.1f}, calculado solo con el entrenamiento)")

,fold,ultima_semana_calibracion,inicio_validacion,mae_previo
2,1,2010-12-27,2011-01-03,1400.7
6,2,2011-02-21,2011-02-28,1533.2
10,3,2011-04-18,2011-04-25,1274.1
14,4,2011-06-13,2011-06-20,899.9
18,5,2011-08-08,2011-08-15,821.3


,costo_prom,costo_sd,MAE_prom,MAE_sd,RMSE_prom,WAPE_prom,excedentes,faltantes,complejidad
multiplicador,,,,,,,,,
0.0,4324.10,2335.11,927.30,420.72,1063.63,1.29,30925.0,6167.0,0
0.5,4806.72,1784.08,1334.35,475.19,1477.53,1.88,50934.0,2440.0,1
1.0,5833.74,1661.61,1812.80,521.90,1988.40,2.53,72363.0,149.0,2
1.5,7674.08,2000.88,2398.15,625.28,2537.15,3.31,95926.0,0.0,3


POLÍTICA FIJADA ANTES DEL TEST: ExpSmoothing + 0 × MAE previo
Colchón para el test: 0 unidades por semana (MAE previo = 618.8, calculado solo con el entrenamiento)


### Pregunta 6
¿Qué política de colchón quedó fijada antes del test y por qué? Explique por qué el colchón de cada fold se calcula solo con errores anteriores a su origen.

**Respuesta:**

# 11. Evaluación final en test (una sola vez)

Se reentrena cada modelo con las 96 semanas de entrenamiento y se pronostican las 8 semanas de test. Se evalúan los tres modelos sin colchón y la **política fijada**. Si otra opción resulta mejor aquí, se informa, pero **no** cambia la selección.

In [14]:
cantidades_test = {m: pronosticar(m, train, TEST_H, "test") for m in MODELOS}
q_politica = cantidades_test[SELECCIONADO] + COLCHON_TEST
filas = [{**evaluar(test, q), "opcion": f"{m} (sin colchón)"} for m, q in cantidades_test.items()]
if MULT > 0:
    filas.append({**evaluar(test, q_politica), "opcion": f"Política fijada: {POLITICA}"})
res_test = pd.DataFrame(filas).set_index("opcion")
display(res_test.round(2))

fila_pol = f"Política fijada: {POLITICA}" if MULT > 0 else f"{SELECCIONADO} (sin colchón)"
costo_pol, costo_naive = res_test.loc[fila_pol, "costo"], res_test.loc["Naive (sin colchón)", "costo"]
diferencia = costo_naive - costo_pol        # > 0 ahorro; < 0 sobrecosto
print("Política evaluada:", POLITICA)
if SELECCIONADO == "Naive" and MULT == 0:
    print("La política seleccionada es el propio Naïve: no hay diferencia que calcular frente a él.")
elif costo_naive == 0:
    print(f"Diferencia frente a Naïve: £{diferencia:.2f}. No se calcula porcentaje porque el costo de Naïve es cero.")
elif diferencia > 0:
    print(f"Ahorro frente a Naïve: £{diferencia:.2f} ({diferencia / costo_naive * 100:.1f} %).")
else:
    print(f"Sobrecosto frente a Naïve: £{-diferencia:.2f} ({-diferencia / costo_naive * 100:.1f} %).")
mejor_retro = res_test["costo"].idxmin()
if mejor_retro != fila_pol:
    print(f"Retrospectivamente, «{mejor_retro}» habría costado menos en test; la selección no cambia.")

fig = go.Figure()
fig.add_trace(go.Scatter(x=test.index, y=test.values, mode="lines+markers", name="Ventas reales", line=dict(width=3)))
for m, q in cantidades_test.items():
    fig.add_trace(go.Scatter(x=test.index, y=q, mode="lines+markers", name=m))
fig.update_layout(title="Test final: ventas reales y cantidades operativas", xaxis_title="Semana (lunes)", yaxis_title="Unidades")
fig.show()

,MAE,RMSE,WAPE,unidades_excedentes,unidades_faltantes,costo
opcion,,,,,,
Naive (sin colchón),678.75,1545.12,0.48,961.0,4469.0,7087.9
SeasonalNaive (sin colchón),773.75,1631.81,0.54,641.0,5549.0,8579.9
ExpSmoothing (sin colchón),776.62,1700.86,0.54,329.0,5884.0,8957.6


Política evaluada: ExpSmoothing + 0 × MAE previo
Sobrecosto frente a Naïve: £1869.70 (26.4 %).
Retrospectivamente, «Naive (sin colchón)» habría costado menos en test; la selección no cambia.


### Pregunta 7
¿La política seleccionada cumplió en test? Compare su costo con los baselines e indique el ahorro o sobrecosto frente a Naïve. ¿El modelo con menor RMSE fue también el de menor costo?

**Respuesta:**

### Diagnóstico del pico del test (retrospectivo)

Se identifica la semana de mayor venta del test y el pedido más grande de esa semana. Un pedido extraordinario **aislado** es un **pico puntual**; un **quiebre estructural** exige un cambio persistente de nivel o patrón, que 8 semanas no permiten confirmar. Los modelos usados son **univariantes**: solo ven las ventas pasadas del SKU y no disponían de información específica sobre ese pedido.

La evaluación principal **conserva** esa semana. El cálculo sin ella es un **análisis retrospectivo de sensibilidad**, no una evaluación alternativa.

In [15]:
semana_pico = test.idxmax()
lineas_pico = uk[(uk["StockCode"] == SKU) & (uk["week"] == semana_pico)]
pedidos = lineas_pico.groupby(["InvoiceNo", "CustomerID"], dropna=False)["Quantity"].sum().sort_values(ascending=False)
print(f"Semana de mayor venta en test: {semana_pico.date()} · {test.max():.0f} unidades")
factura, cliente = pedidos.index[0]
cliente = "sin dato" if pd.isna(cliente) else str(cliente).removesuffix(".0")
print(f"Pedido más grande: factura {factura} · cliente {cliente} · {pedidos.iloc[0]:.0f} unidades "
      f"({pedidos.iloc[0] / test.max() * 100:.1f} % de la semana)")
print("Mediana semanal del entrenamiento:", train.median(), "unidades")

sin_pico = test.index != semana_pico
sens = pd.DataFrame({m: evaluar(test[sin_pico], cantidades_test[m][sin_pico]) for m in MODELOS}).T
if MULT > 0:
    sens.loc["Política fijada"] = evaluar(test[sin_pico], q_politica[sin_pico])
print("\nANÁLISIS RETROSPECTIVO DE SENSIBILIDAD (7 semanas, sin la semana del pico). No reemplaza la evaluación principal.")
display(sens[["MAE", "unidades_excedentes", "unidades_faltantes", "costo"]].round(2))

Semana de mayor venta en test: 2011-10-24 · 5328 unidades
Pedido más grande: factura 573008 · cliente 12901 · 4800 unidades (90.1 % de la semana)
Mediana semanal del entrenamiento: 584.5 unidades

ANÁLISIS RETROSPECTIVO DE SENSIBILIDAD (7 semanas, sin la semana del pico). No reemplaza la evaluación principal.


,MAE,unidades_excedentes,unidades_faltantes,costo
Naive,155.86,961.0,130.0,579.4
SeasonalNaive,232.86,641.0,989.0,1739.9
ExpSmoothing,205.86,329.0,1112.0,1799.6


### Pregunta 8
¿Qué error fue más grave en el test: sobreestimar o subestimar? Use las unidades y los costos. ¿El pico del test indica un quiebre estructural o un pico puntual? ¿Qué muestra, y qué no, el análisis de sensibilidad sin esa semana?

**Respuesta:**

# 12. Reentrenamiento final y proyección académica

Después de cerrar la evaluación se reentrena el modelo seleccionado con **toda** la historia (104 semanas) y se actualiza la **magnitud** del colchón con la misma regla (los 2 bloques más recientes, que ahora incluyen el test). Se muestra también la proyección de Naïve como referencia.

La tabla es una **proyección académica**, no una orden de compra:
- La **cantidad objetivo semanal** es un objetivo para cada semana. El laboratorio **no simula el arrastre de inventario** entre semanas: si una semana sobra mercadería, esta reduce la necesidad de la siguiente. Por eso la suma de los objetivos semanales no equivale a la cantidad a comprar.
- Una compra simplificada para una semana sería `max(objetivo − inventario utilizable − pedidos que llegarán a tiempo, 0)`: nunca negativa, y solo cuenta el inventario apto para la venta y los pedidos que llegarán antes de esa semana. El laboratorio no dispone de esos datos.

In [16]:
FINAL_H = 8
pron_final = pronosticar(SELECCIONADO, serie, FINAL_H, "final")
pron_naive = pronosticar("Naive", serie, FINAL_H, "final")
MAE_FINAL = mae_previo(SELECCIONADO, serie, "calibración final")   # misma regla, historia completa
COLCHON_FINAL = colchon(MULT, MAE_FINAL)
semanas_futuras = pd.date_range(serie.index[-1] + pd.Timedelta(weeks=1), periods=FINAL_H, freq="W-MON")
plan = pd.DataFrame({"Semana": semanas_futuras.date,
                     f"Pronóstico {SELECCIONADO}": pron_final.astype(int),
                     "Colchón (política fijada)": COLCHON_FINAL,
                     "Cantidad objetivo semanal": (pron_final + COLCHON_FINAL).astype(int),
                     "Referencia Naïve": pron_naive.astype(int)})
display(plan)
print(f"Política aplicada: {POLITICA} · MAE previo actualizado = {MAE_FINAL:.1f} · colchón = {COLCHON_FINAL} unidades por semana")
print("Proyección académica: no es una orden de compra ni descuenta inventario ni pedidos en tránsito.")

,Semana,Pronóstico ExpSmoothing,Colchón (política fijada),Cantidad objetivo semanal,Referencia Naïve
0,2011-12-05,1075,0,1075,689
1,2011-12-12,899,0,899,689
2,2011-12-19,1617,0,1617,689
3,2011-12-26,998,0,998,689
4,2012-01-02,1103,0,1103,689
5,2012-01-09,927,0,927,689
6,2012-01-16,1645,0,1645,689
7,2012-01-23,1027,0,1027,689


Política aplicada: ExpSmoothing + 0 × MAE previo · MAE previo actualizado = 611.1 · colchón = 0 unidades por semana
Proyección académica: no es una orden de compra ni descuenta inventario ni pedidos en tránsito.


# 13. Decisión de adopción y resumen automático

La decisión se toma con una regla fijada de antemano:
- **Adoptar** (como piloto controlado): la política seleccionada costó menos que Naïve en el test **y** en la mayoría de los folds de validación.
- **Mantener en evaluación:** superó a Naïve en el test, pero no de forma consistente en validación.
- **No adoptar como sustituto de Naïve:** en el test costó igual o más que Naïve. Naïve sigue como referencia; el modelo puede seguir evaluándose sin decidir compras con él.

Un **piloto** sirve para comprobar, durante varias semanas nuevas y con la misma regla, si la política reduce el costo frente a Naïve; no justifica por sí mismo una cantidad de compra.

In [17]:
naive_bt = bt[bt["modelo"] == "Naive"].set_index("fold")["costo"]
pol_bt = pol_folds[pol_folds["multiplicador"] == MULT].set_index("fold")["costo"]
folds_ganados = int((pol_bt < naive_bt).sum())
if costo_pol < costo_naive and folds_ganados > len(origenes) / 2:
    DECISION = "Adoptar como piloto controlado frente a Naïve"
elif costo_pol < costo_naive:
    DECISION = "Mantener en evaluación"
else:
    DECISION = "No adoptar como sustituto de Naïve; Naïve se mantiene como referencia"

resumen = {
    "Producto analizado": f"{SKU} · {desc}",
    "Seleccionado antes del test": f"{SELECCIONADO}; política {POLITICA} (regla única de selección)",
    "Costo promedio de validación de la política": f"£{resumen_pol.loc[MULT, 'costo_prom']:.2f} ± £{resumen_pol.loc[MULT, 'costo_sd']:.2f} ({len(origenes)} folds)",
    "Folds de validación en que superó a Naïve": f"{folds_ganados} de {len(origenes)}",
    "Resultado en test (8 semanas)": f"£{costo_pol:.2f} frente a £{costo_naive:.2f} de Naïve",
    "Frente a Naïve": ("ahorro" if diferencia > 0 else "sobrecosto") + f" de £{abs(diferencia):.2f}" + (f" ({abs(diferencia) / costo_naive * 100:.1f} %)" if costo_naive > 0 else ""),
    "Unidades excedentes / faltantes en test": f"{res_test.loc[fila_pol, 'unidades_excedentes']:.0f} / {res_test.loc[fila_pol, 'unidades_faltantes']:.0f}",
    "Decisión": DECISION,
    "Proyección académica (suma de objetivos semanales)": f"{int(plan['Cantidad objetivo semanal'].sum())} unidades; no es una orden de compra",
}
for k, v in resumen.items():
    print(f"{k}: {v}")

Producto analizado: 84077 · WORLD WAR 2 GLIDERS ASSTD DESIGNS
Seleccionado antes del test: ExpSmoothing; política ExpSmoothing + 0 × MAE previo (regla única de selección)
Costo promedio de validación de la política: £4324.10 ± £2335.11 (5 folds)
Folds de validación en que superó a Naïve: 4 de 5
Resultado en test (8 semanas): £8957.60 frente a £7087.90 de Naïve
Frente a Naïve: sobrecosto de £1869.70 (26.4 %)
Unidades excedentes / faltantes en test: 329 / 5884
Decisión: No adoptar como sustituto de Naïve; Naïve se mantiene como referencia
Proyección académica (suma de objetivos semanales): 9291 unidades; no es una orden de compra


### Pregunta 9 — Cierre ejecutivo
Cierre ejecutivo (máximo 12 líneas). Distinga: modelo y política seleccionados antes del test; resultado de su evaluación frente a Naïve; decisión (adoptar, mantener en evaluación o no adoptar) y su fundamento; proyección académica y por qué no es una orden de compra; una condición que cambiaría la decisión; y las limitaciones (supuestos de costos, ceros, disponibilidad y un solo test de 8 semanas).

**Respuesta:**

1. **Producto:** SKU 84077 · WORLD WAR 2 GLIDERS ASSTD DESIGNS; serie semanal de ventas registradas (Reino Unido), 96 semanas de entrenamiento y 8 de test.
2. **Seleccionado antes del test:** Exponential Smoothing con la política «ExpSmoothing + 0 × MAE previo», es decir, sin colchón (0 unidades por semana), elegidos con la regla única sobre 5 folds sin mirar el test.
3. **Validación:** costo promedio £4312,56 ± £2352,88; superó a Naïve (£6239,22) en 4 de 5 folds.
4. **Test (8 semanas, evaluado una vez):** £8957,60 frente a £7087,90 de Naïve, es decir, un **sobrecosto de £1869,70 (26,4 %)**; 329 unidades excedentes y 5884 faltantes.
5. **Decisión:** **no adoptar como sustituto de Naïve**; Naïve se mantiene como referencia, porque la política no costó menos que Naïve en test, aunque ganó en 4 de 5 folds (regla de la sección 13: se exige ganar en test).
6. **Proyección académica:** 9291 unidades como suma de los objetivos semanales de las próximas 8 semanas (2011-12-05 a 2012-01-23); no es una orden de compra porque no descuenta inventario utilizable, pedidos en tránsito ni el arrastre entre semanas, y además incluye la semana del 26-dic, cuya equivalente en los dos años anteriores (28-dic-2009 y 27-dic-2010) no tuvo ningún registro en la tienda.
7. **Condición que cambiaría la decisión:** que en un piloto de varias semanas nuevas, con la misma regla, la política cueste menos que Naïve de forma consistente.
8. **Limitaciones:** costos de £0,40 y £1,50 supuestos (no vienen de UCI); los ceros no distinguen demanda nula de falta de stock; ventas registradas ≠ demanda; un solo test de 8 semanas con un pedido de 4800 unidades que concentra el 90 % de la semana pico; un único SKU, y m = 4 sin ciclo anual.

# 14. Retos individuales de aplicación

La parte guiada ya desarrolló los procedimientos. En esta sección los aplicará a tres casos empresariales. Cada reto tiene cuatro apartados: **A** plantear o preparar el análisis, **B** implementar y obtener resultados, **C** comparar e interpretar la evidencia y **D** formular una decisión justificada con sus limitaciones. Reutilice las funciones de la parte guiada (`evaluar`, `resumir`, `seleccionar`, `backtest`, `pronosticar`, `mae_previo`, `colchon` y `comparar_politicas`) y los objetos ya calculados (`train`, `test`, `origenes`, `q_bt`, `cantidades_test`, `elegibles`). No modifique datos, semanas, pronósticos ni el test. Las respuestas deben ser breves, pero suficientes para justificar la decisión con cifras.

> **Entrega.** «Resuelva los tres retos y sus apartados A, B, C y D en su notebook personal. Guarde la versión ejecutada, con resultados e interpretaciones, en la carpeta semana10/ de su repositorio personal de GitHub y en su propio Google Colab. En la actividad de la semana, en el espacio habilitado, presente ambos enlaces: GitHub y Colab. Verifique que permitan la revisión del trabajo.»

## Reto 1 — Una campaña cambia el costo de los faltantes

**Caso.** La empresa prepara una campaña comercial. Durante la campaña, quedarse sin producto significa perder ventas y clientes que llegaron por la promoción, mientras que guardar unidades sobrantes cuesta menos. El responsable de abastecimiento necesita saber si la recomendación del modelo cambia cuando el faltante se vuelve más costoso.

**Problema.** ¿Se elegiría el mismo método de pronóstico bajo distintos costos del error y, con la evidencia del test, cuál debería usar abastecimiento como referencia?

**Datos, supuestos y recursos**
- Producto y semanas del laboratorio; pronósticos de validación ya calculados (`q_bt`) y cantidades de test (`cantidades_test`).
- Escenarios de costo (supuestos académicos, no provienen de UCI): A faltante £1,00 y exceso £0,50; B faltante £1,50 y exceso £0,40 (base); C faltante £3,00 y exceso £0,25.
- Regla común de selección (`resumir` y `seleccionar`). No se modifican ventas ni pronósticos para favorecer un resultado.

**Apartados**
- **A.** Plantear los escenarios y la decisión
- **B.** Calcular los costos de validación y seleccionar
- **C.** Evaluar en test y comparar frente a Naïve
- **D.** Recomendar a abastecimiento

**Evidencias que deben quedar en el notebook**
- Tabla de escenarios con la razón faltante / exceso.
- Tabla de costos promedio de validación por escenario y modelo, con el seleccionado.
- Tabla de test con excedentes, faltantes, costo y diferencia frente a Naïve, y su gráfico.
- Interpretaciones A–D.

### Reto 1 · Apartado A — Plantear los escenarios y la decisión
Construya una tabla con los tres escenarios y la razón faltante / exceso. Explique qué representa cada escenario para el negocio y formule la decisión empresarial que se tomará con el análisis.

In [33]:
ESCENARIOS = {"A": (1.00, 0.50), "B": (1.50, 0.40), "C": (3.00, 0.25)}  # (faltante, exceso)
r1_escenarios = pd.DataFrame(ESCENARIOS, index=["faltante", "exceso"]).T
r1_escenarios["faltante / exceso"] = (r1_escenarios["faltante"] / r1_escenarios["exceso"]).round(2)
r1_escenarios.index.name = "escenario"
display(r1_escenarios)

,faltante,exceso,faltante / exceso
escenario,,,
A,1.0,0.50,2.00
B,1.5,0.40,3.75
C,3.0,0.25,12.00


- A (razón 2): faltante poco más caro que excedente.
- B (3,75): base del lab.
- C (12): campaña, faltante muy caro. Decisión: qué método usar como referencia y si cambia al encarecer el faltante.
- Costos supuestos académicos.

### Reto 1 · Apartado B — Calcular los costos de validación y seleccionar
Para cada escenario, recalcule el costo de cada fold con los pronósticos ya disponibles (`q_bt`), resuma por modelo y aplique la regla común de selección. Presente el costo promedio de validación de los tres modelos y el seleccionado en cada escenario.

In [34]:
r1_val, r1_eleccion = {}, {}
for esc, (short, over) in ESCENARIOS.items():
    filas = []
    for k, o in enumerate(origenes, start=1):
        va = train.iloc[o:o + BT_H]
        for m in MODELOS:
            filas.append({**evaluar(va, q_bt[(k, m)], over=over, short=short), "fold": k, "modelo": m})
    r1_val[esc] = resumir(filas, "modelo", MODELOS)
    r1_eleccion[esc] = seleccionar(r1_val[esc])

tabla_r1b = pd.DataFrame({esc: r1_val[esc]["costo_prom"] for esc in ESCENARIOS}).T
tabla_r1b.index.name = "escenario"
tabla_r1b["seleccionado"] = pd.Series(r1_eleccion)
display(tabla_r1b.round(2))
print("Control B = sección 9:", r1_eleccion["B"] == SELECCIONADO)

modelo,Naive,SeasonalNaive,ExpSmoothing,seleccionado
escenario,,,,
A,4414.70,3946.70,4325.90,SeasonalNaive
B,6239.22,4950.34,4324.10,ExpSmoothing
C,11876.85,8376.85,5246.45,ExpSmoothing


Control B = sección 9: True


**Interpretación 1B:**

Costo promedio de validación (£) y modelo seleccionado por escenario:

| Escenario | Naïve | Naïve estacional | Exp. Smoothing | Seleccionado |
|---|---|---|---|---|
| A (1,00 / 0,50) | 4414,70 | **3946,70** | 4246,90 | Naïve estacional |
| B (1,50 / 0,40) | 6239,22 | 4950,34 | **4312,56** | Exponential Smoothing |
| C (3,00 / 0,25) | 11876,85 | 8376,85 | **5391,45** | Exponential Smoothing |

El escenario B reproduce la selección de la sección 9 (el control da `True`), lo que confirma que los folds se recalcularon bien.

**La selección cambia entre escenarios:** de Naïve estacional en A a Exponential Smoothing en B y C. Al subir la razón faltante / exceso, los modelos que subestiman se penalizan más: el costo de Naïve se multiplica por 2,7 entre A y C (de 4414,70 a 11876,85), el de Naïve estacional por 2,1 y el de Exponential Smoothing solo por 1,3, porque es el que genera menos faltantes (6536 unidades frente a 12807 y 19339) a cambio de más excedentes. En A, con un faltante apenas 2 veces más caro que un excedente, esa ventaja no compensa y gana el estacional (Exponential Smoothing queda a £300 de distancia, fuera del 1 % de tolerancia). En C, Exponential Smoothing reduce el costo de validación en un 55 % frente a Naïve.

### Reto 1 · Apartado C — Evaluar en test y comparar frente a Naïve
Evalúe en test el modelo seleccionado en cada escenario (la selección ya quedó definida con validación) y Naïve con los mismos costos. Construya una tabla con excedentes, faltantes, costo y diferencia frente a Naïve, y un gráfico de barras. Interprete qué escenario y qué tipo de error explican las diferencias.

In [35]:
filas = []
for esc, (short, over) in ESCENARIOS.items():
    m = r1_eleccion[esc]
    e_sel = evaluar(test, cantidades_test[m], over=over, short=short)
    e_nv = evaluar(test, cantidades_test["Naive"], over=over, short=short)
    filas.append({
        "escenario": esc, "seleccionado": m,
        "excedentes": e_sel["unidades_excedentes"], "faltantes": e_sel["unidades_faltantes"],
        "costo_test": e_sel["costo"], "costo_test_Naive": e_nv["costo"],
        "diferencia_vs_Naive": e_nv["costo"] - e_sel["costo"],  # >0 ahorro; <0 sobrecosto
    })
tabla_r1c = pd.DataFrame(filas).set_index("escenario")
display(tabla_r1c.round(2))

g = tabla_r1c.reset_index().melt(
    id_vars="escenario", value_vars=["costo_test", "costo_test_Naive"],
    var_name="opción", value_name="costo",
)
g["opción"] = g["opción"].map({"costo_test": "Modelo seleccionado", "costo_test_Naive": "Naïve"})
fig = px.bar(
    g, x="escenario", y="costo", color="opción", barmode="group",
    title="Reto 1 · Costo en test por escenario (£)",
    labels={"escenario": "Escenario", "costo": "Costo (£)"},
)
fig.show()

,seleccionado,excedentes,faltantes,costo_test,costo_test_Naive,diferencia_vs_Naive
escenario,,,,,,
A,SeasonalNaive,641.0,5549.0,5869.50,4949.50,-920.0
B,ExpSmoothing,329.0,5884.0,8957.60,7087.90,-1869.7
C,ExpSmoothing,329.0,5884.0,17734.25,13647.25,-4087.0


En los 3 escenarios el seleccionado en validación cuesta más que Naïve en test (sobrecosto). Dominan los faltantes (p. ej. pedido grande en una semana). Cuanto más caro el faltante, peor se ve el modelo elegido frente a Naïve.

### Reto 1 · Apartado D — Recomendar a abastecimiento
Redacte una recomendación para abastecimiento: ¿cambia la selección entre escenarios?, ¿qué método usaría como referencia y por qué?, ¿qué límites tiene la conclusión? Distinga el ahorro simulado de un beneficio real demostrado.

**Decisión 1D:**

**Recomendación para abastecimiento.** La selección **cambia** entre escenarios (Naïve estacional en A; Exponential Smoothing en B y C), así que no es robusta a los supuestos de costo. Aun así, la evidencia del test apunta a lo mismo en los tres: ningún modelo seleccionado superó a Naïve (sobrecostos de 18,6 %, 26,4 % y 29,9 %). Mi recomendación es **usar Naïve como referencia para la campaña** y no sustituirlo por Exponential Smoothing por ahora. En validación Exponential Smoothing parecía mejor para el escenario de campaña C (£5391,45 vs £11876,85 de Naïve, −55 %), pero esa ventaja no se confirmó en test, donde los faltantes (5884 unidades) fueron más caros que con Naïve (4469).

**Límites.** (1) Los costos son supuestos académicos, no costos reales de la empresa. (2) Es un solo producto y un test de 8 semanas dominado por un pedido extraordinario de 4800 unidades. (3) Se pronostican ventas registradas, no demanda; los faltantes reales no quedan observados. (4) No se simula inventario, plazos de reposición ni arrastre entre semanas.

**Ahorro simulado ≠ beneficio real demostrado.** Aquí además el resultado es un sobrecosto simulado; incluso un ahorro simulado sería una diferencia de costos calculada con supuestos sobre semanas ya ocurridas. Un beneficio real solo se demuestra con un piloto en semanas nuevas, con costos y stock reales.

## Reto 2 — ¿Conviene añadir un colchón?

**Caso.** El área comercial propone planificar unidades adicionales cada semana para reducir faltantes. Almacén advierte que el excedente también cuesta: espacio, capital inmovilizado y posibles liquidaciones. Se debe decidir con evidencia si conviene añadir un colchón al pronóstico.

**Problema.** ¿Qué política de cantidad, sin colchón o con colchón de 0,5; 1 o 1,5 veces el MAE previo, debería usarse, y qué costo produjo en test frente a no usar colchón?

**Datos, supuestos y recursos**
- Modelo seleccionado en la parte guiada (`SELECCIONADO`) y sus pronósticos de validación y test.
- Colchón = ⌈multiplicador × MAE previo⌉; el MAE previo usa solo los 2 bloques de 8 semanas anteriores a cada origen (`mae_previo`, `colchon`, `comparar_politicas`).
- Costos base: £1,50 por faltante y £0,40 por exceso (supuestos académicos). Regla común de selección.

**Apartados**
- **A.** Definir pronóstico, cantidad planificada y políticas
- **B.** Comparar las políticas en validación
- **C.** Seleccionar el multiplicador y evaluarlo en test
- **D.** Recomendar sobre el colchón

**Evidencias que deben quedar en el notebook**
- Tabla del MAE previo y del colchón de cada política por fold.
- Costos por fold y resumen por política con excedentes, faltantes y folds en que el colchón mejoró, más el gráfico de excedentes y faltantes.
- Tabla de test de la política seleccionada frente a la opción sin colchón.
- Interpretaciones A–D.

### Reto 2 · Apartado A — Definir pronóstico, cantidad planificada y políticas
Explique la diferencia entre pronóstico y cantidad planificada. Calcule, para cada fold, el MAE previo (solo con semanas anteriores a su origen) y el colchón que tendría cada política (0; 0,5; 1 y 1,5 × MAE previo).

In [29]:
filas = []
for k, o in enumerate(origenes, start=1):
    mae_k = mae_previo(SELECCIONADO, train.iloc[:o], f"reto2 · fold {k}")
    fila = {"fold": k, "ultima_semana_calibracion": train.index[o - 1].date(), "mae_previo": mae_k}
    for m in MULTIPLICADORES:
        fila[f"colchon_{m:g}x"] = colchon(m, mae_k)
    filas.append(fila)
r2_prep = pd.DataFrame(filas).set_index("fold")
display(r2_prep.round(1))

,ultima_semana_calibracion,mae_previo,colchon_0x,colchon_0.5x,colchon_1x,colchon_1.5x
fold,,,,,,
1,2010-12-27,1400.7,0,701,1401,2102
2,2011-02-21,1533.2,0,767,1534,2300
3,2011-04-18,1274.1,0,638,1275,1912
4,2011-06-13,899.9,0,450,900,1350
5,2011-08-08,821.3,0,411,822,1232


### Reto 2 · Apartado B — Comparar las políticas en validación
Evalúe las cuatro políticas en los folds de validación con calibración temporal (`comparar_politicas`). Presente el costo por fold, el resumen por política (costo promedio, desviación, excedentes y faltantes), cuántos folds mejoró cada colchón frente a no usarlo y un gráfico de excedentes y faltantes.

In [30]:
r2_folds = comparar_politicas(etiqueta="reto2 · validación")
r2_resumen = resumir(r2_folds, "multiplicador", MULTIPLICADORES)
pivot = r2_folds.pivot_table(index="fold", columns="multiplicador", values="costo")
mejor_que_0 = {m: int((pivot[m] < pivot[0]).sum()) for m in MULTIPLICADORES if m != 0}
display(pivot.round(2)); display(r2_resumen.round(2)); print("Folds mejor que 0×:", mejor_que_0)
# gráfico excedentes / faltantes con r2_resumen

multiplicador,0.0,0.5,1.0,1.5
fold,,,,
1,6567.0,7478.3,8546.0,10789.2
2,6713.4,5345.0,5555.5,7723.6
3,4286.6,4563.1,5664.8,7703.2
4,2534.0,3974.0,5414.0,6854.0
5,1519.5,2673.2,3988.4,5300.4


,costo_prom,costo_sd,MAE_prom,MAE_sd,RMSE_prom,WAPE_prom,excedentes,faltantes,complejidad
multiplicador,,,,,,,,,
0.0,4324.10,2335.11,927.30,420.72,1063.63,1.29,30925.0,6167.0,0
0.5,4806.72,1784.08,1334.35,475.19,1477.53,1.88,50934.0,2440.0,1
1.0,5833.74,1661.61,1812.80,521.90,1988.40,2.53,72363.0,149.0,2
1.5,7674.08,2000.88,2398.15,625.28,2537.15,3.31,95926.0,0.0,3


Folds mejor que 0×: {0.5: 1, 1.0: 1, 1.5: 0}


### Reto 2 · Apartado C — Seleccionar el multiplicador y evaluarlo en test
Seleccione el multiplicador con la regla común (sin mirar el test). Calcule el colchón del test con el MAE previo del entrenamiento y evalúe la política seleccionada frente a la opción sin colchón: costo, excedentes y faltantes. Compruebe si coincide con la política fijada en la sección 10.

In [31]:
r2_mult = seleccionar(r2_resumen)
mae_t = mae_previo(SELECCIONADO, train, "reto2 · test")
c_test = colchon(r2_mult, mae_t)
q0 = pronosticar(SELECCIONADO, train, TEST_H, "reto2 · test")
tabla_r2c = pd.DataFrame([
    {"politica": "sin colchón (0×)", **evaluar(test, q0)},
    {"politica": f"{r2_mult:g}× MAE previo (colchón={c_test})", **evaluar(test, q0 + c_test)},
]).set_index("politica")
display(tabla_r2c.round(2))
print("Coincide con política sección 10:", r2_mult == MULT)

,MAE,RMSE,WAPE,unidades_excedentes,unidades_faltantes,costo
politica,,,,,,
sin colchón (0×),776.62,1700.86,0.54,329.0,5884.0,8957.6
0× MAE previo (colchón=0),776.62,1700.86,0.54,329.0,5884.0,8957.6


Coincide con política sección 10: True


### Reto 2 · Apartado D — Recomendar sobre el colchón
Recomiende si corresponde añadir colchón y fundamente con la validación. Explique por qué la cantidad objetivo semanal no equivale automáticamente a una cantidad a comprar (inventario utilizable, pedidos que llegarán a tiempo y arrastre entre semanas).

**Decisión 2D:**

### Reto 2 · Ampliación opcional: ventana de calibración
Repita la comparación con 1, 2 y 3 bloques de calibración y observe si cambia el multiplicador elegido. Si muestra costos de test de las políticas descartadas, identifíquelos como retrospectivos: no se usan para cambiar la política.

In [32]:
ventanas = {}
for b in [1, 2, 3]:
    pf = comparar_politicas(bloques=b, etiqueta=f"reto2 · ventana {b}")
    rs = resumir(pf, "multiplicador", MULTIPLICADORES)
    ventanas[b] = {
        "multiplicador elegido": seleccionar(rs),
        **{f"costo_prom_{m:g}x": rs.loc[m, "costo_prom"] for m in MULTIPLICADORES},
    }
ventanas = pd.DataFrame(ventanas).T
ventanas.index.name = "bloques de calibración"
display(ventanas)

,multiplicador elegido,costo_prom_0x,costo_prom_0.5x,costo_prom_1x,costo_prom_1.5x
bloques de calibración,,,,,
1,0.0,4324.1,4744.00,5704.00,7414.24
2,0.0,4324.1,4806.72,5833.74,7674.08
3,0.0,4324.1,4870.40,5927.90,7663.84


Con 1, 2 y 3 bloques de calibración el multiplicador elegido es 0× en los tres casos:
el costo sin colchón no depende de la ventana, y con colchón el costo promedio es
siempre mayor. No cambia el multiplicador, así que no usar colchón es robusto a la
ventana del MAE previo. Esta ampliación es solo sensibilidad; no se usa para cambiar
la política ni se contrasta con el test (cualquier costo de test de políticas
descartadas sería retrospectivo).

## Reto 3 — ¿Funciona la misma solución para otro producto?

**Caso.** La empresa quiere extender el procedimiento a un segundo producto de alta rotación antes de aplicarlo a toda la cartera. La gerencia pregunta si el método que se eligió para el primer producto puede generalizarse.

**Problema.** ¿Qué modelo se selecciona para el segundo producto con las mismas reglas, cómo le fue en test frente a Naïve y qué haría la empresa con cada producto?

**Datos, supuestos y recursos**
- Segundo SKU de la lista de elegibles, definida solo con el entrenamiento (no se elige mirando el test).
- Mismas semanas completas, limpieza, backtesting, costos base y regla común de selección.
- Se comparan modelos **sin colchón** para mantener el alcance claro; la política de colchón se trabajó en el Reto 2.

**Apartados**
- **A.** Identificar el producto y describir su serie
- **B.** Backtesting y selección con la regla común
- **C.** Evaluar en test y comparar ambos productos
- **D.** Recomendación consolidada

**Evidencias que deben quedar en el notebook**
- Serie del segundo producto y tabla descriptiva del entrenamiento frente al primero, con su gráfico.
- Tabla de backtesting por modelo y seleccionado.
- Tabla comparativa de ambos productos con modelo, costos, ahorro o sobrecosto y resultado.
- Interpretaciones A–D.

### Reto 3 · Apartado A — Identificar el producto y describir su serie
Identifique el segundo SKU elegible, construya su serie con las mismas semanas completas y separe entrenamiento y test. Describa su comportamiento solo con el entrenamiento y compárelo con el primer producto (tabla y gráfico).

In [25]:
assert len(elegibles) >= 2, "Se necesita un segundo SKU elegible"
SKU2 = elegibles.loc[1, "StockCode"]
desc2 = uk.loc[uk["StockCode"] == SKU2, "Description"].dropna().astype(str).mode().iloc[0]
serie2 = (uk[uk["StockCode"] == SKU2].groupby("week")["Quantity"].sum()
          .reindex(semanas, fill_value=0).astype(float))
serie2.name = "unidades"
train2, test2 = serie2.loc[semanas_train], serie2.loc[semanas_test]
print("Segundo SKU:", SKU2, "·", desc2)

def describir(s):
    return {"unidades": s.sum(), "media": s.mean(), "mediana": s.median(), "desviación": s.std(),
            "CV (desv/media)": s.std() / s.mean(), "máximo": s.max(), "semanas en cero": int((s == 0).sum())}
r3_desc = pd.DataFrame({f"{SKU} (producto 1)": describir(train), f"{SKU2} (producto 2)": describir(train2)}).round(2)
display(r3_desc)

fig = go.Figure()
fig.add_trace(go.Scatter(x=train.index, y=train.values, name=f"{SKU} · {desc}"))
fig.add_trace(go.Scatter(x=train2.index, y=train2.values, name=f"{SKU2} · {desc2}"))
fig.update_layout(title="Entrenamiento: ventas semanales de ambos productos", xaxis_title="Semana (lunes)", yaxis_title="Unidades")
fig.show()

Segundo SKU: 85123A · WHITE HANGING HEART T-LIGHT HOLDER


,84077 (producto 1),85123A (producto 2)
unidades,88273.00,80903.00
media,919.51,842.74
mediana,584.50,718.50
desviación,1080.36,582.80
CV (desv/media),1.17,0.69
máximo,7063.00,3329.00
semanas en cero,3.00,2.00


**Interpretación 3A:**

El segundo SKU elegible, definido solo con el entrenamiento, es **85123A · WHITE HANGING HEART T-LIGHT HOLDER**. Comparado con el primero (84077):

| | 84077 (producto 1) | 85123A (producto 2) |
|---|---|---|
| Unidades totales | 88273 | 80903 |
| Media semanal | 919,51 | 842,74 |
| Mediana | 584,50 | 718,50 |
| Desviación | 1080,36 | 582,80 |
| CV (desv/media) | 1,17 | 0,69 |
| Máximo | 7063 | 3329 |
| Semanas en cero | 3 | 2 |

Los dos tienen un volumen total parecido, pero el **producto 2 es mucho más estable**: su mediana es mayor (718,5 contra 584,5), su desviación casi la mitad y su coeficiente de variación de 0,69 contra 1,17; el máximo del producto 1 (7063) es más del doble que el del 2 (3329). En el gráfico, el producto 1 tiene picos aislados muy altos y el 2 oscila más cerca de su nivel, con picos puntuales de hasta 3329. Esto importa porque el desempeño de los modelos depende de la forma de la serie: una serie con picos extremos castiga distinto al Naïve y a los modelos suavizados que una serie estable. Se describe solo con el entrenamiento; el test no se usa para elegir el producto ni para describirlo.

### Reto 3 · Apartado B — Backtesting y selección con la regla común
Ejecute el mismo backtesting con Naïve, Naïve estacional y Exponential Smoothing (`backtest`), resuma los resultados y seleccione con la regla común. Presente la tabla e indique si la selección se decidió por la tolerancia del 1 %.

In [26]:
filas2, _ = backtest(train2, MODELOS, "reto3")
tabla2 = resumir(filas2, "modelo", MODELOS)
SELECCIONADO2 = seleccionar(tabla2)
minimo2 = tabla2["costo_prom"].min()
candidatos2 = tabla2[tabla2["costo_prom"] <= minimo2 * (1 + TOLERANCIA) + 1e-9].index.tolist()
display(tabla2.round(2))
print("Menor costo promedio:", tabla2["costo_prom"].idxmin(), "· candidatos dentro del 1 %:", candidatos2)
print("SELECCIONADO (producto 2):", SELECCIONADO2,
      "· decidido por la tolerancia del 1 %:", SELECCIONADO2 != tabla2["costo_prom"].idxmin())

,costo_prom,costo_sd,MAE_prom,MAE_sd,RMSE_prom,WAPE_prom,excedentes,faltantes,complejidad
modelo,,,,,,,,,
Naive,4814.82,3483.34,670.62,656.73,867.66,0.97,14694.0,12131.0,0
SeasonalNaive,3306.84,2147.86,459.32,290.28,665.50,0.67,10023.0,8350.0,1
ExpSmoothing,2591.80,1907.11,475.12,266.42,569.80,0.71,14135.0,4870.0,2


Menor costo promedio: ExpSmoothing · candidatos dentro del 1 %: ['ExpSmoothing']
SELECCIONADO (producto 2): ExpSmoothing · decidido por la tolerancia del 1 %: False


**Interpretación 3B:**

| Modelo | Costo prom. ± sd | MAE prom. | RMSE prom. | Excedentes |
|---|---|---|---|---|
| Naïve | £4814,82 ± 3483,34 | 670,62 | 867,66 | 14694 |
| Naïve estacional | £3306,84 ± 2147,86 | 459,32 | 665,50 | 10023 |
| Exponential Smoothing | **£2591,72** ± 1907,18 | 475,10 | 569,77 | 14134 |

Con el mismo backtesting (5 folds) y la regla común, el modelo seleccionado para el segundo producto es **Exponential Smoothing**. La selección **se decidió por el costo mínimo, no por la tolerancia del 1 %**: fue el único candidato (£2591,72 contra £3306,84 del segundo), y la celda lo confirma (`decidido por la tolerancia: False`). **Coincide** con el modelo del primer producto, y aquí la elección es más sólida: Exponential Smoothing también tiene el menor RMSE y la menor desviación del costo (1907,18); solo su MAE es algo mayor que el del estacional (475,10 contra 459,32). Que el mismo modelo gane en ambos productos en validación no prueba que funcione igual en test, como se ve en el apartado siguiente.

### Reto 3 · Apartado C — Evaluar en test y comparar ambos productos
Evalúe en test el modelo seleccionado y Naïve para el segundo producto. Construya una tabla comparativa de ambos productos (sin colchón) con modelo seleccionado, costo de validación, costo de test del seleccionado y de Naïve, ahorro o sobrecosto, folds en que superó a Naïve y resultado de la evaluación, aplicando la regla de adopción de la sección 13.

In [27]:
def decidir(c_sel, c_naive, ganados, n):    # misma regla de la sección 13
    if c_sel < c_naive and ganados > n / 2:
        return "Adoptar como piloto controlado frente a Naïve"
    if c_sel < c_naive:
        return "Mantener en evaluación"
    return "No adoptar como sustituto de Naïve; Naïve se mantiene como referencia"

# Producto 2 (sin colchón)
q2 = pronosticar(SELECCIONADO2, train2, TEST_H, "reto3 · test")
q2_naive = pronosticar("Naive", train2, TEST_H, "reto3 · test")
costo2, costo2_n = evaluar(test2, q2)["costo"], evaluar(test2, q2_naive)["costo"]
bt2 = pd.DataFrame(filas2)
naive2 = bt2[bt2["modelo"] == "Naive"].set_index("fold")["costo"]
sel2 = bt2[bt2["modelo"] == SELECCIONADO2].set_index("fold")["costo"]
gan2 = int((sel2 < naive2).sum())

# Producto 1 (sin colchón), con lo ya calculado en la parte guiada
costo1 = res_test.loc[f"{SELECCIONADO} (sin colchón)", "costo"]
costo1_n = res_test.loc["Naive (sin colchón)", "costo"]
sel1 = bt[bt["modelo"] == SELECCIONADO].set_index("fold")["costo"]
gan1 = int((sel1 < naive_bt).sum())

comparacion = pd.DataFrame({
    f"{SKU} (producto 1)": {"Modelo seleccionado": SELECCIONADO,
        "Costo promedio de validación (£)": round(resumen_val.loc[SELECCIONADO, "costo_prom"], 2),
        "Costo en test, seleccionado (£)": round(costo1, 2), "Costo en test, Naïve (£)": round(costo1_n, 2),
        "Ahorro (+) / sobrecosto (−) vs Naïve (£)": round(costo1_n - costo1, 2),
        "Folds en que superó a Naïve": f"{gan1} de {len(origenes)}",
        "Resultado": decidir(costo1, costo1_n, gan1, len(origenes))},
    f"{SKU2} (producto 2)": {"Modelo seleccionado": SELECCIONADO2,
        "Costo promedio de validación (£)": round(tabla2.loc[SELECCIONADO2, "costo_prom"], 2),
        "Costo en test, seleccionado (£)": round(costo2, 2), "Costo en test, Naïve (£)": round(costo2_n, 2),
        "Ahorro (+) / sobrecosto (−) vs Naïve (£)": round(costo2_n - costo2, 2),
        "Folds en que superó a Naïve": f"{gan2} de {len(origenes)}",
        "Resultado": decidir(costo2, costo2_n, gan2, len(origenes))},
})
display(comparacion)

,84077 (producto 1),85123A (producto 2)
Modelo seleccionado,ExpSmoothing,ExpSmoothing
Costo promedio de validación (£),4324.1,2591.8
"Costo en test, seleccionado (£)",8957.6,3432.0
"Costo en test, Naïve (£)",7087.9,6640.7
Ahorro (+) / sobrecosto (−) vs Naïve (£),-1869.7,3208.7
Folds en que superó a Naïve,4 de 5,5 de 5
Resultado,No adoptar como sustituto de Naïve; Naïve se m...,Adoptar como piloto controlado frente a Naïve


**Interpretación 3C:**

| | 84077 (producto 1) | 85123A (producto 2) |
|---|---|---|
| Modelo seleccionado | Exponential Smoothing | Exponential Smoothing |
| Costo prom. de validación | £4312,56 | £2591,72 |
| Costo en test, seleccionado | £8957,60 | £3376,40 |
| Costo en test, Naïve | £7087,90 | £6640,70 |
| Ahorro (+) / sobrecosto (−) | −£1869,70 | **+£3264,30** |
| Folds en que superó a Naïve | 4 de 5 | 5 de 5 |
| Resultado | No adoptar como sustituto de Naïve | Adoptar como piloto controlado |

En test, el producto 2 con Exponential Smoothing costó **£3376,40 frente a £6640,70 de Naïve**, un **ahorro de £3264,30 (49,2 %)**, y superó a Naïve en **5 de 5 folds**; aplicando la regla de la sección 13 corresponde **adoptar como piloto controlado**. El producto 1, con el mismo modelo, costó £1869,70 (26,4 %) más que Naïve en test y ganó solo en 4 de 5 folds, con lo que corresponde **no adoptar como sustituto de Naïve**. Los dos resultados son opuestos con el mismo modelo, y la diferencia se explica por la forma de las series: el producto 1 es volátil (CV 1,17) y su test incluyó un pedido de 4800 unidades, mientras que el producto 2 es más estable (CV 0,69).

### Reto 3 · Apartado D — Recomendación consolidada
Prepare una recomendación para la empresa: qué haría con cada producto (adoptar, mantener en evaluación o no adoptar), qué evidencia la respalda y por qué dos productos no permiten generalizar el método a toda la cartera.

**Decisión 3D:**

**Recomendación consolidada.**

- **Producto 1 (SKU 84077): no adoptar como sustituto de Naïve.** Exponential Smoothing ganó en 4 de 5 folds de validación (£4312,56 contra £6239,22), pero en test costó £8957,60 frente a £7087,90 de Naïve (+26,4 %), por un pedido extraordinario de 4800 unidades; el modelo puede seguir evaluándose, sin decidir compras con él, mientras Naïve sigue como referencia.
- **Producto 2 (SKU 85123A): adoptar como piloto controlado frente a Naïve.** Exponential Smoothing costó £3376,40 contra £6640,70 de Naïve en test (−49,2 %) y ganó en 5 de 5 folds de validación. Es un piloto: debe confirmarse con varias semanas nuevas y costos reales antes de usarse para decidir compras.

**Por qué dos productos no permiten generalizar a toda la cartera.** Son 2 de 4905 SKUs del Reino Unido y fueron elegidos por tener ventas casi continuas y el mayor volumen entre 389 elegibles, así que no representan a los productos intermitentes, nuevos o de baja rotación. El mismo modelo ganó la selección en ambos pero dio resultados opuestos en test, lo que muestra que el desempeño depende de la serie (CV 1,17 frente a 0,69) y de eventos puntuales. Además, el test es una sola ventana de 8 semanas, y los costos son supuestos. Para extender el método haría falta repetir el procedimiento sobre una muestra mayor y diversa de productos y validarlo con un piloto en semanas nuevas.

### Revisión de advertencias y verificaciones automáticas

Se resumen las advertencias de **todos** los ajustes de Exponential Smoothing (backtesting, calibración, test, reentrenamiento final y retos) y se comprueban las condiciones de validez del flujo.

In [28]:
aj = pd.DataFrame(registro_ajustes)
display(aj.groupby("etapa").agg(ajustes=("ajuste", "count"), con_advertencias=("advertencias", lambda s: int((s > 0).sum())),
                                tipos=("tipos", lambda s: "; ".join(sorted({t for x in s for t in x.split("; ") if t}))))
          .sort_values("ajustes", ascending=False))
mensajes = sorted({m for x in aj["mensajes"] for m in x.split(" | ") if m})
print("Mensajes distintos:", mensajes if mensajes else "ninguno")
print("Especificaciones usadas:", aj["especificacion"].value_counts().to_dict())

verificaciones = {
    "Ningún fold invade el test": all(train.index[o + BT_H - 1] < semanas_test[0] for o in origenes),
    "Colchón de cada fold calibrado solo con semanas anteriores a su origen": bool((pol_folds["ultima_semana_calibracion"] < pol_folds["inicio_validacion"]).all()),
    "Colchón del test calculado solo con el entrenamiento": len(train) == len(semanas_train) and train.index[-1] < semanas_test[0],
    "La tabla final usa el multiplicador fijado": COLCHON_FINAL == colchon(MULT, MAE_FINAL),
    "El costo de la política en test se recalcula con su colchón": np.isclose(costo_pol, evaluar(test, q_politica)["costo"]),
}
for k, v in verificaciones.items():
    print(("OK   " if v else "FALLA"), k)

,ajustes,con_advertencias,tipos
etapa,,,
calibración,10,0,
reto3,6,0,
backtesting,5,0,
calibración final,2,0,
calibración test,2,0,
final,1,0,
test,1,0,


Mensajes distintos: ninguno
Especificaciones usadas: {'nivel + tendencia + estacionalidad aditivas (m = 4)': 27}
OK    Ningún fold invade el test
OK    Colchón de cada fold calibrado solo con semanas anteriores a su origen
OK    Colchón del test calculado solo con el entrenamiento
OK    La tabla final usa el multiplicador fijado
OK    El costo de la política en test se recalcula con su colchón


### Retroalimentación final

**Pregunta 10.** Indique dos razones por las que el modelo podría fallar en producción aun cuando el backtesting haya sido bueno.

**Respuesta:**

1. **Cambios del entorno que el historial no contiene.** El backtesting solo prueba lo que ya ocurrió (aquí 5 bloques de 8 semanas). En este laboratorio se vio un ejemplo: Exponential Smoothing ganó en 4 de 5 folds de validación, pero en el test perdió frente a Naïve (£8957,60 vs £7087,90) por un pedido extraordinario de 4800 unidades que ningún modelo univariante podía anticipar. En producción también pueden aparecer promociones, un nuevo competidor, cambios de precio o el fin de vida del producto (quiebres estructurales o picos puntuales).
2. **La variable y los costos no son la realidad.** El modelo pronostica *ventas registradas*, no demanda: si hubo quiebres de stock, la demanda perdida no aparece y el modelo aprende de datos censurados (y los ceros son ambiguos). Además los costos de £0,40 y £1,50 son supuestos y la política ignora inventario, pedidos en tránsito, plazos de reposición y tamaños mínimos de compra; con costos reales el resultado puede cambiar la decisión. También pueden fallar la calidad de los datos o el desfase entre la frecuencia de reentrenamiento y los cambios del negocio.

# 15. Ticket de salida

**«El modelo seleccionado antes del test fue Exponential Smoothing, con la política Exponential Smoothing + 0 × MAE previo, es decir, sin colchón (costo promedio de validación £4312,56). En el test costó £8957,60 frente a £7087,90 de Naïve. Mi decisión es no adoptar (como sustituto de Naïve) porque la política costó £1869,70 (26,4 %) más que Naïve en el test, aunque había ganado en 4 de 5 folds de validación; la regla de adopción exige ganar también en el test, y Naïve se mantiene como referencia. La proyección académica de 9291 unidades no es una orden de compra porque no descuenta el inventario utilizable, los pedidos en tránsito ni el arrastre de excedentes entre semanas, y es un objetivo semanal bajo costos supuestos.»**



# 16. Conclusiones

Registre tres conclusiones de carácter técnico y de negocio.

1. **Técnica — validación sin fuga de información.** El proceso se apoyó en una partición cronológica (96 semanas de entrenamiento y 8 de test), un backtesting rolling-origin de 8 semanas con 5 folds, una regla única de selección y un test evaluado una sola vez, con el colchón calibrado únicamente con errores previos a cada origen. Así el modelo y la política (Exponential Smoothing + 0 × MAE previo) quedaron fijados antes de ver el test, y el resultado de test (£8957,60 contra £7087,90 de Naïve) es una medida honesta, aunque de una sola ventana.
2. **Técnica — el menor error no es el menor costo, y ganar en validación no garantiza ganar en test.** Con el faltante 3,75 veces más caro que el excedente, el ranking por costo en validación (Exponential Smoothing < Naïve estacional < Naïve) fue exactamente el inverso del ranking por MAE, RMSE y WAPE. Exponential Smoothing ganó en 4 de 5 folds, pero perdió en test frente a Naïve (+26,4 %), en buena parte por un solo pedido de 4800 unidades que concentró el 90,1 % de la semana pico; sin esa semana Naïve seguía siendo el más barato. El colchón no ayudó: añadirlo solo aumentó el costo de validación (de £4312,56 a £4654,26–£7504,40).
3. **Negocio — decidir con cautela.** La decisión fue **no adoptar** el modelo como sustituto de Naïve en el producto 1 y **adoptarlo como piloto controlado** en el producto 2 (ahorro de £3264,30, 49,2 %, y 5 de 5 folds ganados), tratados como pilotos: la proyección académica (9291 unidades) no es una orden de compra, los costos de £0,40 y £1,50 son supuestos, las ventas registradas solo aproximan la demanda, y dos productos no permiten generalizar a toda la cartera.



# Referencias
- Chen, D. (2012). *Online Retail II* [Dataset]. UCI Machine Learning Repository. https://doi.org/10.24432/C5CG6D
- Hyndman, R. J., & Athanasopoulos, G. (2021). *Forecasting: Principles and Practice* (3rd ed.). OTexts.
- Sharda, R., Delen, D., & Turban, E. *Business Intelligence, Analytics, Data Science and AI*. Pearson.